In [ ]:
import  osiris_utils as ou
from matplotlib import pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter

plt.rcParams['font.size'] = 14


In [ ]:
def createSimDic(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/dtw{dtw}/{key}.in")
    return sim

In [ ]:
# Normalize axis to w_ce


def _set_scaled_formatter(axis, scale_factor, fmt=".2f"):
    axis.set_major_formatter(
        FuncFormatter(lambda v, pos: f"{v*scale_factor:{fmt}}")
    )

def scale_x_ax(scale_factor, fig, ax, label=r"$t[1 / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_xlabel(label)
    if lock_ticks:
        # freeze current tick positions
        ticks = ax.get_xticks()
        ax.xaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    return fig, ax

def scale_y_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_ylabel(label)
    if lock_ticks:
        ticks = ax.get_yticks()
        ax.yaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    return fig, ax

def scale_z_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_zlabel(label)
    if lock_ticks:
        ticks = ax.get_zticks()
        ax.zaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax

def scale_3d_axes(scale_factor, fig, ax, fmt=".2f"):
    # ax.set_xlabel(r"$x_1[c / \Omega_e]$")
    # ax.set_ylabel(r"$x_2[c / \Omega_e]$")
    # ax.set_zlabel(r"$x_3[c / \Omega_e]$")
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax


In [ ]:
class curvDriftTheo:
    def __init__(self, sim, B=None, rqm = -1, direc = -1):
        self.rqm = rqm
        self.direc = direc
        self.sim = sim
        self.x1_0 = sim["test_electrons"]["tracks"]["x1"][:,0]
        self.x2_0 = sim["test_electrons"]["tracks"]["x2"][:,0]
        self.x3_0 = sim["test_electrons"]["tracks"]["x3"][:,0]
        self.phi0 = np.arctan2(self.x2_0, self.x1_0)
        self.p1_0 = sim["test_electrons"]["tracks"]["p1"][:,0]
        self.p2_0 = sim["test_electrons"]["tracks"]["p2"][:,0]
        self.p3_0 = sim["test_electrons"]["tracks"]["p3"][:,0]
        self.gamma_0 = np.sqrt(1 + self.p1_0**2 + self.p2_0**2 + self.p3_0**2)
        
        self.R0 = np.sqrt(sim["test_electrons"]["tracks"]["x1"][:,0]**2 + sim["test_electrons"]["tracks"]["x2"][:,0]**2)

        if B is not None:
            self.B0 = B
        else:
            self.B0 = np.sqrt(sim["test_electrons"]["tracks"]["B1"][:,0]**2 + sim["test_electrons"]["tracks"]["B2"][:,0]**2 + sim["test_electrons"]["tracks"]["B3"][:,0]**2)

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)
    def b2(self, x1, x2, x3):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)
    def b3(self, x1, x2, x3):
        return np.zeros_like(x1)

    def _curv_v(self):
        sim = self.sim


        p_par0 = (self.p1_0 * self.b1(self.x1_0, self.x2_0, self.x3_0) + \
                self.p2_0 * self.b2(self.x1_0, self.x2_0, self.x3_0) + \
                self.p3_0 * self.b3(self.x1_0, self.x2_0, self.x3_0) ) / self.B0
        
        v_par = p_par0 / self.gamma_0
        vc = self.rqm * p_par0**2 / self.B0 / self.gamma_0 * self.direc / self.R0

        return vc, v_par

    def get_curv_traj(self, t):
        t = np.asarray(t)              # shape: (nt,)
        omega = self.v_par / self.R0      # shape: (npart,)

        x3 = self.x3_0[:, None] + np.outer(self.vc, t)

        phase = np.outer(omega, t) + self.phi0[:, None]

        x1 = self.R0[:, None] * np.cos(phase)
        x2 = self.R0[:, None] * np.sin(phase)

        return np.array([x1, x2, x3])
    

In [ ]:
test = "Curv"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim = createSimDic(path, sim_labels, test)

In [ ]:
t = sim["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
traj_theo = curvDriftTheo(sim["Gca"]["1000"], B, direc=1).get_curv_traj(t)[:,:,0]
radius_theo = curvDriftTheo(sim["Gca"]["1000"], B, direc=1).R0
print("theo radius:", radius_theo)
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

from matplotlib.lines import Line2D

fig, ax = plt.subplots()
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        radius = np.sqrt(x1**2 + x2**2)

        err = np.abs(radius - radius_theo)/ radius_theo

        mean = np.mean(err)
        std = np.std(err, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    eb = ax.errorbar(
        X, Y, yerr=YERR,
        label=fr"{pusher}",
        fmt='o',
        markersize=5,
        linestyle=linestyle,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

    ax.plot(
        X,
        YMax,
        marker='x',
        linestyle='none',
        markersize=5,
        markeredgewidth=1.0,
        zorder=3,
        color=eb.lines[0].get_color(),
    )

ax.set_ylabel("relative radial error")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
ax.legend()
ax.set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        # num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        # ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_ylabel(fr"{components[i]} error dt L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_DoublePrec"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaHighRes"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaOnlyCurvDrift"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradB"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBInit"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBInit"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBInit = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradB"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradB = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBdrift"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBdrift = createSimDic(path, sim_labels, test)

In [ ]:
test_sims = {
    "All drifts": sim_Curv_1step,
    "No grad B init": sim_Curv_1step_GcaNoGradBInit,
    "No grad B drift": sim_Curv_1step_GcaNoGradBdrift,
    "Neither": sim_Curv_1step_GcaNoGradB,
}

pusher = "Gca"
grid = test_sims["All drifts"][pusher]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for test_label, sim in test_sims.items():
    X = []
    Y = []
    YERR = []
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = float(dtw.replace("_", ".")) / 1000
        err = (np.abs(traj - traj_theo)) / L / num_steps

        err_radial = np.sqrt(err[0]**2 + err[1]**2)
        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    order = np.argsort(X)
    X = X[order]
    Y = Y[order]
    YERR = YERR[order]
    YMax = YMax[order]

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=test_label,
            fmt='o',
            markersize=5,
            linestyle='-',
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend(title="Test")
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GradBDiag"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
sim["Gca"]["1000"]["test_electrons"]["tracks"]["dB1dx1"]

def get_dBdx(sim):
    return np.array([
        [
            sim["test_electrons"]["tracks"]["dB1dx1"],
            sim["test_electrons"]["tracks"]["dB1dx2"],
            sim["test_electrons"]["tracks"]["dB1dx3"],
        ],
        [
            sim["test_electrons"]["tracks"]["dB2dx1"],
            sim["test_electrons"]["tracks"]["dB2dx2"],
            sim["test_electrons"]["tracks"]["dB2dx3"],
        ],
        [
            sim["test_electrons"]["tracks"]["dB3dx1"],
            sim["test_electrons"]["tracks"]["dB3dx2"],
            sim["test_electrons"]["tracks"]["dB3dx3"],
        ],
    ])

def get_B(sim):
    return np.array([
        sim["test_electrons"]["tracks"]["B1"],
        sim["test_electrons"]["tracks"]["B2"],
        sim["test_electrons"]["tracks"]["B3"],
    ])

def get_traj(sim):
    return np.array([
        sim["test_electrons"]["tracks"]["x1"],
        sim["test_electrons"]["tracks"]["x2"],
        sim["test_electrons"]["tracks"]["x3"],
    ])

B_theo = 1000
dbdx = get_dBdx(sim["Gca"]["1000"])
traj = get_traj(sim["Gca"]["1000"])
B_vec = get_B(sim["Gca"]["1000"])
B_mag = np.linalg.norm(B_vec, axis=0)

gradB = np.einsum("jnt,jint->int", B_vec, dbdx) / B_mag
gradB_mag = np.linalg.norm(gradB, axis=0)

In [ ]:
gradB_mag_normalized = gradB_mag / B_theo / B_theo

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    traj[0].flatten(),
    traj[1].flatten(),
    c=gradB_mag_normalized.flatten(),
    s=2,
    cmap="viridis"
)
plt.colorbar(sc, label=r"$|\nabla B| / B^2 \, [e / c^2 m_e]$")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    traj[0].flatten(),
    traj[1].flatten(),
    c=np.abs(B_mag.flatten()-B_theo)/B_theo,
    s=2,
    cmap="viridis"
)
plt.colorbar(sc, label=r"Relative error in $|B|$")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

In [ ]:
test = "Curv_1step_GcaInitCurvOnly"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaOnlyCurvDriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaGradBOnly"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBInitV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1stepV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1stepV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBInitV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBInit = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradB = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBdriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBdrift = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaOnlyCurvDriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaOnlyCurvDrift = createSimDic(path, sim_labels, test)

In [ ]:
test_sims = {
    "All drifts": sim_Curv_1step,
    "No grad B init": sim_Curv_1step_GcaNoGradBInit,
    "No grad B drift": sim_Curv_1step_GcaNoGradBdrift,
    "Neither": sim_Curv_1step_GcaNoGradB,
    "Only curv drift": sim_Curv_1step_GcaOnlyCurvDrift,
}

pusher = "Gca"
grid = test_sims["All drifts"][pusher]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for test_label, sim in test_sims.items():
    X = []
    Y = []
    YERR = []
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = float(dtw.replace("_", ".")) / 1000
        err = (np.abs(traj - traj_theo)) / L / num_steps

        err_radial = np.sqrt(err[0]**2 + err[1]**2)
        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    order = np.argsort(X)
    X = X[order]
    Y = Y[order]
    YERR = YERR[order]
    YMax = YMax[order]

    for i, ax in enumerate(axes):
        if test_label == "All drifts" or test_label == "No grad B init":
            linestyle = '-'
        elif test_label == "Only curv drift":
            linestyle = ':'
        else:            linestyle = '--'

        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=test_label,
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend(title="Test")
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoDrifts"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
# test = "Curv"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim = createSimDic(path, sim_labels, test)

# test = "Curv_dx"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim2 = createSimDic(path, sim_labels, test)

# sims = {"80": sim, "120": sim2}

In [ ]:


# components = ["r", "x3"]
# fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

# pushers = []
# for dx in sims.keys():
#     for pusher in sims[dx].keys():
#         if pusher not in pushers:
#             pushers.append(pusher)

# color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
# style_map = {"120": "--"}
# for dx in sims.keys():
#     linestyle = style_map.get(dx, '-')
#     t = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
#     traj_theo = curvDriftTheo(sims[dx]["Gca"]["1000"], B).get_curv_traj(t)[:,:,0]

#     grid = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"].grid
#     grid = [float(grid[0, 0]), float(grid[0, 1])]
#     L = grid[1] - grid[0]

#     for pusher in sims[dx].keys():
#         X = []    
#         Y = []
#         YERR = []    
#         YMax = []
#         for dtw in sims[dx][pusher].keys():
#             x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
#             x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
#             x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

#             traj = np.array([x1, x2, x3])

#             err = (np.abs(traj - traj_theo)) / L

#             err_radial = np.sqrt(err[0]**2 + err[1]**2)

#             err = np.array([err_radial, err[2]])

#             mean = np.mean(err, axis=1)
#             std = np.std(err, axis=1, ddof=1)

#             X.append(float(dtw.replace("_", ".")))
#             Y.append(mean)
#             YERR.append(std)
#             YMax.append(np.max(err, axis=1))
        
#         X = np.asarray(X, dtype=float)
#         Y = np.asarray(Y, dtype=float)
#         YERR = np.asarray(YERR, dtype=float)
#         YMax = np.asarray(YMax, dtype=float)

#         for i, ax in enumerate(axes):
#             eb = ax.errorbar(
#                 X, Y[:, i], yerr=YERR[:, i],
#                 fmt='o',
#                 markersize=5,
#                 linestyle=linestyle,
#                 linewidth=1.2,
#                 capsize=3,
#                 elinewidth=1.0,
#                 color=color_map[pusher],
#             )

#             ax.plot(
#                 X,
#                 YMax[:, i],
#                 marker='x',
#                 linestyle='none',
#                 markersize=5,
#                 markeredgewidth=1.0,
#                 zorder=3,
#                 color=eb.lines[0].get_color(),
#             )
#             ax.set_ylabel(fr"{components[i]} error / L")
#             ax.set_xscale("log")
#             ax.set_yscale("log")
#             ax.grid(True, alpha=0.3)

# pusher_handles = [
#     Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
#     for pusher in pushers
# ]
# style_handles = [
#     Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
#     for dx in sims.keys()
# ]

# axes[0].set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
# axes[0].legend(handles=pusher_handles + style_handles)
# axes[-1].set_xlabel("dtw")
# fig.tight_layout()


In [ ]:
# test = "Curv_1step"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim = createSimDic(path, sim_labels, test)

# test = "Curv_dx_1step"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim2 = createSimDic(path, sim_labels, test)

# sims = {"80": sim, "120": sim2}

In [ ]:
# components = ["r", "x3"]
# fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

# pushers = []
# for dx in sims.keys():
#     for pusher in sims[dx].keys():
#         if pusher not in pushers:
#             pushers.append(pusher)

# color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
# style_map = {"120": "--"}
# for dx in sims.keys():
#     linestyle = style_map.get(dx, '-')

#     for pusher in sims[dx].keys():
#         X = []    
#         Y = []
#         YERR = []    
#         YMax = []
#         for dtw in sims[dx][pusher].keys():
#             t = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
#             traj_theo = curvDriftTheo(sims[dx][pusher][dtw], B).get_curv_traj(t)[:,:,0]

#             grid = sims[dx][pusher][dtw]["test_electrons"]["tracks"].grid
#             grid = [float(grid[0, 0]), float(grid[0, 1])]
#             L = grid[1] - grid[0]

#             x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
#             x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
#             x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

#             traj = np.array([x1, x2, x3])

#             err = (np.abs(traj - traj_theo)) / L

#             err_radial = np.sqrt(err[0]**2 + err[1]**2)

#             err = np.array([err_radial, err[2]])

#             mean = np.mean(err, axis=1)
#             std = np.std(err, axis=1, ddof=1)

#             X.append(float(dtw.replace("_", ".")))
#             Y.append(mean)
#             YERR.append(std)
#             YMax.append(np.max(err, axis=1))
        
#         X = np.asarray(X, dtype=float)
#         Y = np.asarray(Y, dtype=float)
#         YERR = np.asarray(YERR, dtype=float)
#         YMax = np.asarray(YMax, dtype=float)

#         for i, ax in enumerate(axes):
#             eb = ax.errorbar(
#                 X, Y[:, i], yerr=YERR[:, i],
#                 fmt='o',
#                 markersize=5,
#                 linestyle=linestyle,
#                 linewidth=1.2,
#                 capsize=3,
#                 elinewidth=1.0,
#                 color=color_map[pusher],
#             )

#             ax.plot(
#                 X,
#                 YMax[:, i],
#                 marker='x',
#                 linestyle='none',
#                 markersize=5,
#                 markeredgewidth=1.0,
#                 zorder=3,
#                 color=eb.lines[0].get_color(),
#             )
#             ax.set_ylabel(fr"{components[i]} error / L")
#             ax.set_xscale("log")
#             ax.set_yscale("log")
#             ax.grid(True, alpha=0.3)

# pusher_handles = [
#     Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
#     for pusher in pushers
# ]
# style_handles = [
#     Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
#     for dx in sims.keys()
# ]

# axes[0].set_title("1st step")
# axes[0].legend(handles=pusher_handles + style_handles)
# axes[-1].set_xlabel("dtw")
# fig.tight_layout()


### Gca - no GradB

In [ ]:
sim = ou.Simulation("/home/exxxx5/Tese/Decks/StudyConvergence/Curv_GcaNoGradB/Gca/dtw1/Gca.in")
sim["test_electrons"]["tracks"].load_all()

particle = 0

grid = sim["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

theo = curvDriftTheo(sim, 1000, direc = 1)

traj_theo = theo.get_curv_traj(sim["test_electrons"]["tracks"]["t"][0,:])[:,particle,:]

In [ ]:
t = sim["test_electrons"]["tracks"]["t"][particle, :]

traj_sim = np.array([
    sim["test_electrons"]["tracks"]["x1"][particle, :],
    sim["test_electrons"]["tracks"]["x2"][particle, :],
    sim["test_electrons"]["tracks"]["x3"][particle, :],
])

err = np.abs(traj_sim - traj_theo) / L
err_tot = np.sqrt(np.sum((traj_sim - traj_theo)**2, axis=0)) / L

components = ["x1", "x2", "x3"]
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

for i, comp in enumerate(components):
    axes[i].plot(t, err[i], label=fr"$|{comp}_{{sim}} - {comp}_{{theo}}| / L$")
    axes[i].set_ylabel(fr"{comp} err / L")
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

axes[3].plot(t, err_tot, color="black", label=r"$||\mathbf{x}_{sim} - \mathbf{x}_{theo}|| / L$")
axes[3].set_ylabel(r"total err / L")
axes[3].set_xlabel("t")
axes[3].grid(True, alpha=0.3)
axes[3].legend()

fig.suptitle(f"Particle {particle} trajectory error over time")
fig.tight_layout()


In [ ]:
t = sim["test_electrons"]["tracks"]["t"][particle, 1:]
p1 = sim["test_electrons"]["tracks"]["p1"][particle, 1:]
p2 = sim["test_electrons"]["tracks"]["p2"][particle, 1:]
p3 = sim["test_electrons"]["tracks"]["p3"][particle, 1:]
ptotal = np.sqrt(p1**2 + p2**2 + p3**2)

momenta = [p1, p2, p3, ptotal]
labels = ["p1", "p2", "p3", "|p|"]

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

for ax, p, label in zip(axes, momenta, labels):
    ax.plot(t, p, label=label)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("t")
fig.suptitle(f"Particle {particle} momentum over time")
fig.tight_layout()


In [ ]:
# p3 error
t = sim["test_electrons"]["tracks"]["t"][particle, 1:]
p3_sim = sim["test_electrons"]["tracks"]["p3"][particle, 1:]



vc, v_par = theo._curv_v()

p3_theo = vc[particle] * theo.gamma_0[particle]

p3_err = np.abs(p3_sim - p3_theo)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(t, p3_sim, label=r"$p_{3,sim}$")
axes[0].axhline(p3_theo, color="black", linestyle="--", label=fr"$p_{{3,theo}} = {p3_theo:.6g}$")
axes[0].set_ylabel(r"$p_3$")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t, p3_err, color="tab:red", label=r"$|p_{3,sim} - p_{3,theo}|$")
axes[1].set_xlabel("t")
axes[1].set_ylabel(r"$p_3$ error")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle(f"Particle {particle} p3 comparison")
fig.tight_layout()
